# E10 Guatemala · material suplementario reproducible

Este cuaderno ejecuta el flujo completo sin contener el manuscrito. Mantiene separados: (1) los controles E10 publicados, (2) la actualización abierta de emisiones, (3) la reconstrucción económica con abastecimiento importado y (4) las transiciones hipotéticas a E15 y E20.

Autores del suplemento: Juan Alejandro Osorio, Silvia Patricia Villatoro y Noé Salguero.

## 1. Preparar el entorno

En Colab se clonan este repositorio y la dependencia MIP fijada. En una ejecución local, use las carpetas existentes.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    raiz = Path('/content/etanol-e10-guatemala-suplemento-reproducible')
    mip = Path('/content/mip-guatemala-2013-reproducible')
    if not raiz.exists():
        subprocess.run(['git', 'clone', 'https://github.com/JA-Osorio/etanol-e10-guatemala-suplemento-reproducible.git', str(raiz)], check=True)
    dependencia = json.loads((raiz / '03_configuracion' / 'dependencia_mip.json').read_text())
    if not mip.exists():
        subprocess.run(['git', 'clone', dependencia['repositorio'] + '.git', str(mip)], check=True)
    commit_mip = ''.join(dependencia['commit_fijado_partes'])
    subprocess.run(['git', '-C', str(mip), 'checkout', commit_mip], check=True)
else:
    raiz = Path.cwd()
    if raiz.name == '05_cuaderno_colab':
        raiz = raiz.parent
    mip = raiz.parent / 'mip-guatemala-2013-reproducible'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(raiz)], check=True)
print('Suplemento:', raiz)
print('MIP:', mip)

## 2. Ejecutar el único script maestro

Este comando regenera datos procesados, resultados, figuras, controles y manifiesto.

In [ ]:
comando = [sys.executable, str(raiz / '04_reproduccion_python' / 'reproducir_todo.py'), '--mip-dir', str(mip)]
subprocess.run(comando, check=True)

## 3. Verificar las emisiones del artículo

La tabla siguiente contiene únicamente los controles E10 publicados. Las mezclas superiores están en un archivo separado.

In [ ]:
import pandas as pd
from IPython.display import display, Image

publicadas = pd.read_csv(raiz / '06_resultados' / 'emisiones' / 'reproduccion_publicada.csv')
display(publicadas)
display(Image(filename=str(raiz / '06_resultados' / 'figuras' / 'reduccion_ttw_por_mezcla.png')))

## 4. Revisar la reconstrucción económica

El escenario central supone abastecimiento importado y, por tanto, choque de demanda final doméstica igual a cero. Los recargos sobre la referencia FOB son sensibilidades ilustrativas, no costos entregados observados.

In [ ]:
malla = pd.read_csv(raiz / '06_resultados' / 'economia' / 'malla_costos_importacion.csv')
columnas = [
    'escenario', 'recargo_entrega_ilustrativo_fraccion',
    'participacion_gasolina_en_p068', 'cambio_costo_servicio_pct',
    'volumen_alcohol_millones_gal',
    'porcentaje_requerimiento_alcohol_cubierto_por_referencia_comercial',
    'mayor_efecto_precio_pct'
]
display(malla.loc[malla['escenario'].eq('E10'), columnas])
agregados = pd.read_csv(raiz / '06_resultados' / 'economia' / 'efectos_precios_mip_agregados.csv')
grupos_clave = ['agricultura_pesca_silvicultura', 'quimica_farmaceutica', 'transporte_logistica', 'servicios_privados']
display(agregados.loc[agregados['escenario'].eq('E10') & agregados['participacion_gasolina_en_p068'].eq(0.45) & agregados['grupo_analitico'].isin(grupos_clave)])
display(Image(filename=str(raiz / '06_resultados' / 'figuras' / 'sensibilidad_costo_servicio.png')))

## 5. Contrafactuales nacionales y alcance normativo

Los tres proxies nacionales se leen por separado y por Q1 millón de demanda final adicional. La actualización factual de 2026 aplica a gasolina regular; la gasolina superior permanece sin estimación hasta disponer de norma y datos verificables.

In [ ]:
contrafactuales = pd.read_csv(raiz / '06_resultados' / 'economia' / 'contrafactuales_domesticos_normalizados.csv')
transiciones = pd.read_csv(raiz / '06_resultados' / 'escenarios' / 'transiciones_emisiones_normalizadas.csv')
display(contrafactuales)
display(transiciones.loc[transiciones['familia_analitica'].eq('actualizacion_normativa_2026')])

## 6. Actualizar en el futuro

1. Revise primero `00_fuentes_y_trazabilidad/guia_actualizacion.md`.
2. Extraiga una revisión EIA con `04_reproduccion_python/actualizar_serie_eia.py`.
3. Actualice precios solo si son comparables en fecha, mercado y condición de entrega.
4. Para una MIP nueva, cree otra configuración y concordancia; no sobrescriba la versión 2013.
5. No publique una nueva versión si algún control deja de indicar `PASS`.